In [30]:
import os
from PIL import Image

DATASET_DIR = "dataset"

def fix_image(path):
    try:
        img = Image.open(path)

        # Convert images with palette/alpha → JPG
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        # Convert webp/png to jpg
        if path.lower().endswith((".webp", ".png")):
            new_path = path.rsplit(".", 1)[0] + ".jpg"
            img.save(new_path, "JPEG")
            os.remove(path)
            print(f"Converted: {path} → {new_path}")

        else:
            # Just verify JPG
            img.verify()

    except Exception as e:
        print("Corrupted or unreadable → Removing:", path)
        os.remove(path)

# Run on all folders
for folder in ["tomato", "Radish", "Carrot", "Potato"]:
    subdir = os.path.join(DATASET_DIR, folder)
    for file in os.listdir(subdir):
        fix_image(os.path.join(subdir, file))

print("Dataset cleaned successfully!")


Dataset cleaned successfully!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np

# -------------------------
# 1. DATASET PATH
# -------------------------
dataset_path = "dataset"   # your folder structure

# -------------------------
# 2. IMAGE DATA GENERATOR
# -------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

num_classes = len(train_data.class_indices)
print("Detected Classes:", train_data.class_indices)

# -------------------------
# 3. LOAD BASE MODEL (TRANSFER LEARNING)
# -------------------------
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
preds = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------
# 4. TRAIN MODEL
# -------------------------
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

# -------------------------
# 5. SAVE MODEL
# -------------------------
model.save("vegetable_classifier.h5")
print("Model Saved Successfully!")


Found 12268 images belonging to 15 classes.
Found 3066 images belonging to 15 classes.
Detected Classes: {'Bitter_Gourd': 0, 'Bottle_Gourd': 1, 'Brinjal': 2, 'Broccoli': 3, 'Cabbage': 4, 'Capsicum': 5, 'Carrot': 6, 'Cauliflower': 7, 'Cucumber': 8, 'Papaya': 9, 'Potato': 10, 'Pumpkin': 11, 'Radish': 12, 'bean': 13, 'tomato': 14}


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,277,199 (8.69 MB)

 Trainable params: 19,215 (75.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

C:\Users\victus\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 278s 718ms/step - accuracy: 0.8657 - loss: 0.5235 - val_accuracy: 0.9439 - val_loss: 0.2468
Epoch 2/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 187s 488ms/step - accuracy: 0.9791 - loss: 0.0997 - val_accuracy: 0.9501 - val_loss: 0.1890
Epoch 3/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 225s 586ms/step - accuracy: 0.9877 - loss: 0.0612 - val_accuracy: 0.9527 - val_loss: 0.1911
Epoch 4/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 432s 1s/step - accuracy: 0.9905 - loss: 0.0460 - val_accuracy: 0.9560 - val_loss: 0.1782
Epoch 5/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 237s 616ms/step - accuracy: 0.9934 - loss: 0.0356 - val_accuracy: 0.9553 - val_loss: 0.1766
Epoch 6/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 219s 571ms/step - accuracy: 0.9929 - loss: 0.0308 - val_accuracy: 0.9596 - val_loss: 0.1636
Epoch 7/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 287s 747ms/step - accuracy: 0.9960 - loss: 0.0237 - val_accuracy: 0.9573 - val_loss: 0.1877
Epoch 8/15
384/384 ━━━━━━━━━━━━━━━━━━━━ 312s 811ms/step - accuracy: 0.9963 - lo

Model Saved Successfully!


In [35]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model("vegetable_classifier.h5")

img_path = r"C:\Users\victus\tomato_recognization\dataset\test2.jpg"   # your test image
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
class_idx = np.argmax(pred)
confidence = np.max(pred)

# Get class names
dataset_path = "dataset"
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator()
train_data = train_datagen.flow_from_directory(dataset_path)
class_names = list(train_data.class_indices.keys())

print("Prediction:", class_names[class_idx])
print("Confidence:", confidence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 949ms/step
Found 15334 images belonging to 15 classes.
Prediction: tomato
Confidence: 0.9969591
